# CA Cities Boundaries


### Step 0: Set-Up
Import the [climakitae](https://github.com/cal-adapt/climakitae) library and other dependencies.

In [1]:
import climakitae as ck

from climakitae.core.data_load import load

import xarray as xr
import pandas as pd
import geopandas as gpd
from shapely.geometry import mapping
import seaborn as sns
import matplotlib.pyplot as plt
import numpy as np
from climakitae.new_core.data_access.boundaries import Boundaries
import intake

### Step 1: Filter out offshore components from coastal cities

In [ ]:
boundaries_shp = gpd.read_file(
    "California_Cities_and_Identifiers_Blue_Version_view_-3024944392275494958.zip"
)

In [ ]:
boundaries_shp.shape

filter out offshore elements

In [ ]:
boundaries_shp_filtered = boundaries_shp[boundaries_shp['OFFSHORE'].isna()]

In [ ]:
boundaries_shp_filtered['OFFSHORE'].unique()

### Step 2: Convert boundaries shapefile to parquet

In [ ]:
boundaries_shp_filtered.to_parquet("CA_cities_boundaries_filtered.parquet")

### Step 3: Update new core boundaries file

In [36]:
catalog = intake.open_catalog('boundaries.yaml')
boundaries = Boundaries(catalog)
# Get all boundary options for UI population
boundary_options = boundaries.boundary_dict()

In [ ]:
# Access specific boundary data (loaded lazily)
ca_cities = boundaries._ca_cities
ca_counties= boundaries._ca_counties

In [ ]:
ca_counties[ca_counties["NAME"] == "Los Angeles County"]

In [ ]:
ca_counties["NAME"].duplicated().sum()

In [ ]:
ca_cities["CDT_NAME_S"].duplicated().sum()

In [ ]:
boundary_options['CA cities']

### Step 4: Test clip processor

In [9]:
catalog = intake.open_catalog("boundaries.yaml")
boundaries = Boundaries(catalog)
# Get all boundary options for UI population
boundary_options = boundaries.boundary_dict()

In [10]:
cd = ck.ClimateData()  # only give warnings and errors

2026-10-01 14:41:25 - climakitae.new_core.user_interface - INFO - Initializing ClimateData interface
2026-10-01 14:41:25 - climakitae.new_core.dataset_factory - INFO - DatasetFactory initialized with 4 validators and 12 processors
2026-10-01 14:41:25 - climakitae.new_core.user_interface - INFO - ClimateData initialization successful
2026-10-01 14:41:25 - climakitae.new_core.user_interface - INFO - ✅ Ready to query!


In [11]:
cd.show_boundary_options("CA cities")

2026-10-01 14:41:26 - climakitae.new_core.user_interface - INFO - Invalid boundary type, see below for available `boundary_type` options.
2026-10-01 14:41:26 - climakitae.new_core.user_interface - INFO - -----------------------------------------------------------------------
2026-10-01 14:41:26 - climakitae.new_core.user_interface - INFO - CA Census Tracts
2026-10-01 14:41:26 - climakitae.new_core.user_interface - INFO - CA Electric Balancing Authority Areas
2026-10-01 14:41:26 - climakitae.new_core.user_interface - INFO - CA Electric Load Serving Entities (IOU & POU)
2026-10-01 14:41:26 - climakitae.new_core.user_interface - INFO - CA Electricity Demand Forecast Zones
2026-10-01 14:41:26 - climakitae.new_core.user_interface - INFO - CA counties
2026-10-01 14:41:26 - climakitae.new_core.user_interface - INFO - CA watersheds
2026-10-01 14:41:26 - climakitae.new_core.user_interface - INFO - Tribal Areas
2026-10-01 14:41:26 - climakitae.new_core.user_interface - INFO - lat/lon
2026-10-01 

In [38]:
cd.show_variable_id_options()

AttributeError: 'ClimateData' object has no attribute 'show_variable_id_options'

In [29]:
print(cd.__dict__)
print(dir(cd))

{'_log_file': None, '_verbosity': -1, '_factory': <climakitae.new_core.dataset_factory.DatasetFactory object at 0x710ef70ffce0>, '_query': {'catalog': UNSET, 'installation': UNSET, 'activity_id': UNSET, 'institution_id': UNSET, 'source_id': UNSET, 'experiment_id': UNSET, 'table_id': UNSET, 'grid_label': UNSET, 'variable_id': UNSET, 'station_id': UNSET, 'network_id': UNSET, 'processes': UNSET}, 'var_desc':                      variable_id downscaling_method timescale        unit  \
0                             t2          Dynamical    hourly           K   
1                           prec          Dynamical    hourly          mm   
2                     rh_derived          Dynamical    hourly  [0 to 100]   
3             wind_speed_derived          Dynamical    hourly       m s-1   
4         wind_direction_derived          Dynamical    hourly     degrees   
..                           ...                ...       ...         ...   
82                            cf          Dynamical 

In [ ]:
wl_clipped_data = (
    cd.catalog("cadcat")
    .activity_id("WRF")
    .institution_id("UCLA")
    .table_id("day")
    .grid_label("d03")
    .variable_id("t2")  # Temperature at 2 meters
    .processes(
        {
            "warming_level": {
                "warming_levels": [1.5, 2.0, 3.0],  # Warming levels in °C
            },
            "clip": "Alhambra",  # Clip to specified coordinates
        }
    )
    .get()
)

wl_clipped_data

In [ ]:
clip_counties = (
    cd.catalog("cadcat")
    .activity_id("WRF")
    .institution_id("UCLA")
    .table_id("day")
    .grid_label("d03")
    .variable_id("t2")  # Temperature at 2 meters
    .processes(
        {
            "warming_level": {
                "warming_levels": [1.5, 2.0, 3.0],  # Warming levels in °C
            },
            "clip": ["Alameda", "Fresno"],
        }
    )
    .get()
)

clip_counties.isel(sim=0, warming_level=0, time_delta=0).t2.plot(x="lon", y="lat")